# 21 — Persistent Chat Memory (SQLite)

Store conversation history in SQLite for persistence across sessions.

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'

In [ ]:
import sqlite3, tempfile
from datetime import datetime
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser

## SQLite Message Store

In [ ]:
class SQLiteMessageStore:
    def __init__(self, db_path):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("CREATE TABLE IF NOT EXISTS messages (id INTEGER PRIMARY KEY AUTOINCREMENT, session_id TEXT, role TEXT, content TEXT, timestamp TEXT)")
        self.conn.commit()

    def add_message(self, session_id, role, content):
        self.conn.execute("INSERT INTO messages (session_id, role, content, timestamp) VALUES (?,?,?,?)",
                         (session_id, role, content, datetime.now().isoformat()))
        self.conn.commit()

    def get_messages(self, session_id, limit=50):
        rows = self.conn.execute("SELECT role, content FROM messages WHERE session_id=? ORDER BY id DESC LIMIT ?",
                                (session_id, limit)).fetchall()
        rows.reverse()
        return [HumanMessage(content=c) if r == "human" else AIMessage(content=c) for r, c in rows]

    def list_sessions(self):
        return self.conn.execute("SELECT session_id, COUNT(*) as n, MIN(timestamp) FROM messages GROUP BY session_id").fetchall()

    def close(self): self.conn.close()

## Chat with Persistent Memory

In [ ]:
db_path = os.path.join(tempfile.gettempdir(), "langchain_memory_demo.db")
store = SQLiteMessageStore(db_path)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise."),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])
chain = prompt | llm | StrOutputParser()

def chat(session_id, user_input):
    history = store.get_messages(session_id)
    response = chain.invoke({"input": user_input, "history": history})
    store.add_message(session_id, "human", user_input)
    store.add_message(session_id, "ai", response)
    return response

print("--- Session: alice-001 ---")
for msg in ["Hi, I'm Alice. I work on machine learning.", "My current project is a recommendation engine.", "What do you remember about me?"]:
    print(f"User: {msg}\nAI: {chat('alice-001', msg)}\n")

print("--- Session: bob-001 ---")
for msg in ["Hey, I'm Bob. I'm learning Python.", "Can you suggest a project for beginners?"]:
    print(f"User: {msg}\nAI: {chat('bob-001', msg)}\n")

print("--- Resuming alice-001 ---")
print(f"AI: {chat('alice-001', 'Remind me what project I told you about?')}")

store.close()
os.unlink(db_path)